# Sentiment & Clarity Scoring Pipeline

Objective:
- Run FinBERT sentiment scoring and clarity feature computation on every transcript in our buyback announcements dataset.
- Output: a scored CSV with all metrics per transcript.

### Setup & Data Loading

In [4]:
import pandas as pd
import numpy as np
import re
import os
import time
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import sent_tokenize
import nltk

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

os.makedirs('../outputs/eda', exist_ok=True)
os.makedirs('../data/interim', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'teal': '#1A9E8F', 'navy': '#1B2A3D', 'red': '#C0392B',
          'purple': '#7C3AED', 'orange': '#E67E22', 'green': '#27AE60'}

# Load main dataset
df = pd.read_csv('../data/interim/buyback_announcements_only.csv')
print(f"Loaded {len(df)} transcripts, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(df.head(2))

# Try to load Q&A component data
COMPONENTS_PATH = '../data/interim/wrds_transcript_components.csv'
FALLBACK_PATH = '../data/interim/transcript_components.csv'

components_df = None
if os.path.exists(COMPONENTS_PATH):
    components_df = pd.read_csv(COMPONENTS_PATH)
    print(f"Loaded components: {len(components_df)} rows")
    print(f"Component columns: {list(components_df.columns)}")
elif os.path.exists(FALLBACK_PATH):
    components_df = pd.read_csv(FALLBACK_PATH)
    print(f"Loaded fallback components: {len(components_df)} rows")
else:
    print("WARNING: No transcript components file found. Q&A relevance and section-level sentiment will be skipped.")

HAS_COMPONENTS = components_df is not None

Loaded 11313 transcripts, 6 columns
Columns: ['companyid', 'keydevid', 'announceddate', 'mostimportantdateutc', 'relevant_text_only', 'classification']
   companyid    keydevid        announceddate mostimportantdateutc  \
0     403347  84235245.0  2010-01-13 00:00:00           2010-01-20   
1    3289679  96282595.0  2010-01-13 00:00:00           2010-01-26   

                                  relevant_text_only classification  
0  ...d deposits, deferred taxes and other long-t...   ANNOUNCEMENT  
1  ...ins. Through sales leverage, our productivi...   ANNOUNCEMENT  
Loaded components: 9987872 rows
Component columns: ['transcriptid', 'componentorder', 'componenttext', 'transcriptcomponenttypeid', 'speakertypeid', 'speakername', 'component_source', 'companyid', 'companyname', 'ticker', 'call_date']


### Buyback Sentence Extraction

In [5]:
BUYBACK_KEYWORDS = [
    r'\bbuyback\b', r'\bbuy\s*back\b', r'\brepurchas\w*\b',
    r'\bshare\s+repurchas\w*\b', r'\bstock\s+repurchas\w*\b',
    r'\brepurchase\s+program\b', r'\brepurchase\s+plan\b',
]
BUYBACK_PATTERN = re.compile('|'.join(BUYBACK_KEYWORDS), re.IGNORECASE)

# Adapt TEXT_COL to your actual column name
TEXT_COL = 'relevant_text_only'  # Adjusted based on available columns

def extract_buyback_sentences(text):
    if pd.isna(text) or not isinstance(text, str):
        return []
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if BUYBACK_PATTERN.search(s)]

df['buyback_sentences'] = df[TEXT_COL].apply(extract_buyback_sentences)
df['buyback_text'] = df['buyback_sentences'].apply(lambda s: ' '.join(s))
df['buyback_sentence_count'] = df['buyback_sentences'].apply(len)

print(f"Median buyback sentences per call: {df['buyback_sentence_count'].median()}")
print(f"Mean buyback sentences per call: {df['buyback_sentence_count'].mean():.1f}")
print(f"Calls with 0 buyback sentences: {(df['buyback_sentence_count'] == 0).sum()}")

Median buyback sentences per call: 5.0
Mean buyback sentences per call: 5.8
Calls with 0 buyback sentences: 4


### Q&A Section Extraction (if components available)

In [6]:
if HAS_COMPONENTS:
    # Identify key columns in components data
    print(f"Component columns: {list(components_df.columns)}")

    COMP_TYPE_COL = 'transcriptcomponenttypeid'  # adapt if different
    SPEAKER_COL = 'speakertypeid'  # adapt if different
    COMP_TEXT_COL = 'componenttext'  # adapt if different
    TRANSCRIPT_ID_COL = 'transcriptid'  # adapt if different

    # Q&A section = component type 3
    qa_components = components_df[components_df[COMP_TYPE_COL] == 3].copy()
    print(f"Q&A components: {len(qa_components)}")

    # Within Q&A: analyst questions (speaker type 3) vs executive responses (speaker type 1)
    analyst_qs = qa_components[qa_components[SPEAKER_COL] == 3].copy()
    exec_responses = qa_components[qa_components[SPEAKER_COL].isin([1, 2])].copy()
    print(f"Analyst questions: {len(analyst_qs)}")
    print(f"Executive responses: {len(exec_responses)}")

    # Prepared remarks = component type 2
    prep_components = components_df[components_df[COMP_TYPE_COL] == 2].copy()
    print(f"Prepared remarks components: {len(prep_components)}")

    # Aggregate text per transcript for each section
    qa_text_by_transcript = exec_responses.groupby(TRANSCRIPT_ID_COL)[COMP_TEXT_COL].apply(
        lambda x: ' '.join(x.dropna().astype(str))
    ).reset_index().rename(columns={COMP_TEXT_COL: 'qa_exec_text'})

    prep_text_by_transcript = prep_components.groupby(TRANSCRIPT_ID_COL)[COMP_TEXT_COL].apply(
        lambda x: ' '.join(x.dropna().astype(str))
    ).reset_index().rename(columns={COMP_TEXT_COL: 'prep_remarks_text'})

    # Merge into main df
    df_id_col = 'keydevid' if 'keydevid' in df.columns else TRANSCRIPT_ID_COL
    df = df.merge(qa_text_by_transcript, left_on=df_id_col, right_on=TRANSCRIPT_ID_COL, how='left')
    df = df.merge(prep_text_by_transcript, left_on=df_id_col, right_on=TRANSCRIPT_ID_COL, how='left')
    print(f"Transcripts with Q&A text: {df['qa_exec_text'].notna().sum()}")
    print(f"Transcripts with prep text: {df['prep_remarks_text'].notna().sum()}")

    # Extract buyback-specific Q&A responses
    def extract_buyback_qa(text):
        if pd.isna(text) or not isinstance(text, str):
            return ''
        sents = sent_tokenize(text)
        bb_sents = [s for s in sents if BUYBACK_PATTERN.search(s)]
        return ' '.join(bb_sents)

    df['buyback_qa_text'] = df['qa_exec_text'].apply(extract_buyback_qa)

    # Build question-answer pairs for Q&A relevance
    qa_sorted = qa_components.sort_values([TRANSCRIPT_ID_COL, qa_components.columns[0]]).copy()

    def pair_qa(group):
        pairs = []
        last_q = None
        for _, row in group.iterrows():
            if row[SPEAKER_COL] == 3:  # analyst
                last_q = row[COMP_TEXT_COL]
            elif row[SPEAKER_COL] in [1, 2] and last_q is not None:  # executive
                combined = str(last_q) + ' ' + str(row[COMP_TEXT_COL])
                if BUYBACK_PATTERN.search(combined):
                    pairs.append({
                        'question': str(last_q),
                        'answer': str(row[COMP_TEXT_COL])
                    })
                last_q = None
        return pairs

    qa_pairs_by_transcript = qa_sorted.groupby(TRANSCRIPT_ID_COL).apply(pair_qa)
    qa_pairs_dict = qa_pairs_by_transcript.to_dict()
    print(f"Transcripts with buyback Q&A pairs: {sum(1 for v in qa_pairs_dict.values() if v)}")
else:
    print("Skipping Q&A extraction — no components file")
    df['qa_exec_text'] = np.nan
    df['prep_remarks_text'] = np.nan
    df['buyback_qa_text'] = ''
    qa_pairs_dict = {}

Component columns: ['transcriptid', 'componentorder', 'componenttext', 'transcriptcomponenttypeid', 'speakertypeid', 'speakername', 'component_source', 'companyid', 'companyname', 'ticker', 'call_date']
Q&A components: 3281950
Analyst questions: 3281950
Executive responses: 0
Prepared remarks components: 725604
Transcripts with Q&A text: 0
Transcripts with prep text: 0
Transcripts with buyback Q&A pairs: 0


### FinBERT Sentiment Scoring Setup

In [7]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    device=device,
    batch_size=64,
    truncation=True,
    max_length=512
)
print("FinBERT loaded")

def score_sentences(sentences, batch_size=64):
    if not sentences:
        return []
    results = finbert(sentences, batch_size=batch_size)
    scores = []
    for r in results:
        if r['label'] == 'positive':
            scores.append(r['score'])
        elif r['label'] == 'negative':
            scores.append(-r['score'])
        else:
            scores.append(0.0)
    return scores

def sentiment_metrics(scores):
    if not scores or len(scores) == 0:
        return {'mean': np.nan, 'p10': np.nan, 'min': np.nan, 'std': np.nan, 'n': 0}
    arr = np.array(scores)
    return {
        'mean': float(np.mean(arr)),
        'p10': float(np.percentile(arr, 10)),
        'min': float(np.min(arr)),
        'std': float(np.std(arr)),
        'n': len(arr)
    }

CUDA available: True
GPU: NVIDIA GeForce RTX 5080



Device set to use cuda:0


FinBERT loaded


### Score buyback-specific sentences

In [8]:
print(f"Scoring buyback sentences for {len(df)} transcripts...")
start = time.time()

bb_metrics = []
for idx, row in df.iterrows():
    sents = row['buyback_sentences']
    if sents and len(sents) > 0:
        scores = score_sentences(sents)
        m = sentiment_metrics(scores)
    else:
        m = sentiment_metrics([])
    bb_metrics.append(m)

    if (idx + 1) % 2000 == 0:
        elapsed = time.time() - start
        print(f"  {idx+1}/{len(df)} ({elapsed:.0f}s)")

bb_metrics_df = pd.DataFrame(bb_metrics)
df['buyback_sentiment_mean'] = bb_metrics_df['mean']
df['buyback_sentiment_p10'] = bb_metrics_df['p10']
df['buyback_sentiment_min'] = bb_metrics_df['min']
df['buyback_sentiment_std'] = bb_metrics_df['std']
df['buyback_sentiment_n'] = bb_metrics_df['n']

print(f"Buyback sentiment scoring complete in {(time.time()-start)/60:.1f} minutes")
print(f"Mean buyback sentiment: {df['buyback_sentiment_mean'].mean():.4f}")

Scoring buyback sentences for 11313 transcripts...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  2000/11313 (15s)
  4000/11313 (30s)
  6000/11313 (45s)
  8000/11313 (58s)
  10000/11313 (74s)
Buyback sentiment scoring complete in 1.4 minutes
Mean buyback sentiment: 0.3641


### Score full transcripts

In [9]:
FULL_SAMPLE_SIZE = min(5000, len(df))
sample_mask = np.zeros(len(df), dtype=bool)
sample_indices = df.sample(n=FULL_SAMPLE_SIZE, random_state=42).index
sample_mask[sample_indices] = True

print(f"Scoring full transcript sentiment for {FULL_SAMPLE_SIZE} sampled transcripts...")
start = time.time()

full_sent = np.full(len(df), np.nan)
for i, idx in enumerate(sample_indices):
    text = df.loc[idx, TEXT_COL]
    if pd.isna(text) or not isinstance(text, str):
        continue
    sents = sent_tokenize(text)[:200]  # cap at 200 sentences
    if sents:
        scores = score_sentences(sents)
        full_sent[idx] = np.mean(scores)

    if (i + 1) % 500 == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (FULL_SAMPLE_SIZE - i - 1) / rate
        print(f"  {i+1}/{FULL_SAMPLE_SIZE} ({elapsed/60:.1f}min, ~{remaining/60:.1f}min left)")

df['full_transcript_sentiment'] = full_sent
df['sentiment_gap'] = df['buyback_sentiment_mean'] - df['full_transcript_sentiment']

print(f"Full transcript scoring complete in {(time.time()-start)/60:.1f} minutes")

Scoring full transcript sentiment for 5000 sampled transcripts...
  500/5000 (0.2min, ~2.1min left)
  1000/5000 (0.5min, ~1.8min left)
  1500/5000 (0.7min, ~1.6min left)
  2000/5000 (0.9min, ~1.4min left)
  2500/5000 (1.2min, ~1.2min left)
  3000/5000 (1.4min, ~0.9min left)
  3500/5000 (1.6min, ~0.7min left)
  4000/5000 (1.9min, ~0.5min left)
  4500/5000 (2.1min, ~0.2min left)
  5000/5000 (2.3min, ~0.0min left)
Full transcript scoring complete in 2.3 minutes


### Score prepared remarks and Q&A separately

In [10]:
if HAS_COMPONENTS:
    print("Scoring prepared remarks sentiment...")
    start = time.time()

    prep_sent = []
    qa_sent = []
    for idx, row in df.iterrows():
        # Prepared remarks
        pt = row.get('prep_remarks_text', '')
        if isinstance(pt, str) and len(pt) > 20:
            sents = sent_tokenize(pt)[:150]
            scores = score_sentences(sents)
            prep_sent.append(np.mean(scores) if scores else np.nan)
        else:
            prep_sent.append(np.nan)

        # Q&A executive responses
        qt = row.get('qa_exec_text', '')
        if isinstance(qt, str) and len(qt) > 20:
            sents = sent_tokenize(qt)[:150]
            scores = score_sentences(sents)
            qa_sent.append(np.mean(scores) if scores else np.nan)
        else:
            qa_sent.append(np.nan)

        if (idx + 1) % 1000 == 0:
            print(f"  {idx+1}/{len(df)} ({(time.time()-start)/60:.1f}min)")

    df['prep_remarks_sentiment'] = prep_sent
    df['qa_sentiment'] = qa_sent
    print(f"Section sentiment scoring complete in {(time.time()-start)/60:.1f} minutes")
else:
    df['prep_remarks_sentiment'] = np.nan
    df['qa_sentiment'] = np.nan

Scoring prepared remarks sentiment...
  1000/11313 (0.0min)
  2000/11313 (0.0min)
  3000/11313 (0.0min)
  4000/11313 (0.0min)
  5000/11313 (0.0min)
  6000/11313 (0.0min)
  7000/11313 (0.0min)
  8000/11313 (0.0min)
  9000/11313 (0.0min)
  10000/11313 (0.0min)
  11000/11313 (0.0min)
Section sentiment scoring complete in 0.0 minutes


### Clarity Feature 1 — Specificity (0-4)

In [11]:
def compute_specificity(text):
    if pd.isna(text) or not isinstance(text, str) or len(text) < 10:
        return 0
    score = 0
    if re.search(r'\$[\d,.]+\s*(billion|million|B|M|bn|mn)?', text, re.IGNORECASE):
        score += 1
    if re.search(r'[\d,.]+\s*(shares|million\s+shares|billion\s+shares)', text, re.IGNORECASE):
        score += 1
    if re.search(r'(Q[1-4]\s*\d{4}|through\s+\d{4}|over\s+the\s+next|by\s+(year|end|mid)|fiscal\s+year|\d+\s*(year|month|quarter)s?)', text, re.IGNORECASE):
        score += 1
    if re.search(r'(free\s+cash\s+flow|cash\s+on\s+hand|operating\s+cash|debt|credit\s+facility|balance\s+sheet|cash\s+generation|excess\s+cash)', text, re.IGNORECASE):
        score += 1
    return score

# Apply on buyback Q&A text if available, otherwise buyback_text
CLARITY_TEXT_COL = 'buyback_qa_text' if HAS_COMPONENTS and df['buyback_qa_text'].str.len().sum() > 0 else 'buyback_text'
print(f"Computing specificity on column: {CLARITY_TEXT_COL}")

df['specificity'] = df[CLARITY_TEXT_COL].apply(compute_specificity)
print(f"Specificity distribution:\n{df['specificity'].value_counts().sort_index()}")

Computing specificity on column: buyback_text
Specificity distribution:
specificity
0     301
1    3635
2    4642
3    2242
4     493
Name: count, dtype: int64


### Clarity Feature 2 — Hedge Density

In [12]:
HEDGE_WORDS = {
    "may", "might", "could", "possibly", "perhaps", "potentially",
    "uncertain", "uncertainty", "unclear", "unpredictable",
    "approximate", "approximately", "volatile", "volatility",
    "risk", "risks", "risky", "conceivably", "presumably",
    "indefinite", "contingent", "speculative", "tentative",
    "believe", "think", "feel", "hope", "expect", "anticipate",
    "intend", "appear", "suggest", "seem", "likely", "possible",
    "probable", "premature", "eventually",
}

def compute_hedge_density(text):
    if pd.isna(text) or not isinstance(text, str) or len(text) < 10:
        return np.nan
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    if len(words) == 0:
        return np.nan
    hedge_count = sum(1 for w in words if w in HEDGE_WORDS)
    return hedge_count / len(words)

df['hedge_density'] = df[CLARITY_TEXT_COL].apply(compute_hedge_density)
print(f"Hedge density — mean: {df['hedge_density'].mean():.4f}, median: {df['hedge_density'].median():.4f}")

Hedge density — mean: 0.0115, median: 0.0100


### Clarity Feature 3 — Modified FOG

In [13]:
import textstat

FINANCIAL_EXCLUSIONS = {
    "company", "companies", "management", "operating", "financial",
    "securities", "investment", "investments", "performance",
    "opportunity", "opportunities", "infrastructure", "capitalize",
    "approximately", "international", "administration", "environment",
    "executive", "executives", "generation", "acquisition", "acquisitions",
    "technology", "technologies", "significant", "significantly",
    "competitive", "operations", "operational", "continuing",
    "outstanding", "delivered", "delivering", "percentage", "revenue",
    "revenues", "dividend", "dividends", "leverage", "enterprise",
    "inventory", "marketing", "customer", "customers", "improvement",
    "improvements", "industry", "industries", "commercial",
    "manufacture", "manufacturing", "contribute", "contribution",
    "regulatory", "implementing", "implementation", "development",
    "developing", "commitment", "committed", "distributing",
    "distribution", "repurchase", "repurchasing", "authorization",
}

def compute_modified_fog(text):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 30:
        return np.nan
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    total_words = len(words)
    if total_words == 0:
        return np.nan
    sentences = sent_tokenize(text)
    n_sentences = max(len(sentences), 1)
    words_per_sentence = total_words / n_sentences
    complex_count = sum(1 for w in words if textstat.syllable_count(w) >= 3 and w not in FINANCIAL_EXCLUSIONS)
    modified_fog = 0.4 * (words_per_sentence + 100 * complex_count / total_words)
    return modified_fog

df['modified_fog'] = df[CLARITY_TEXT_COL].apply(compute_modified_fog)
df['standard_fog'] = df[CLARITY_TEXT_COL].apply(
    lambda t: textstat.gunning_fog(t) if isinstance(t, str) and len(t) > 30 else np.nan
)
print(f"Standard FOG — mean: {df['standard_fog'].mean():.2f}")
print(f"Modified FOG — mean: {df['modified_fog'].mean():.2f}")
print(f"Reduction: {df['standard_fog'].mean() - df['modified_fog'].mean():.2f} grade levels")

Standard FOG — mean: 15.20
Modified FOG — mean: 15.06
Reduction: 0.14 grade levels


### Clarity Feature 4 — Q&A Relevance (Cosine Similarity)

In [14]:
if HAS_COMPONENTS and qa_pairs_dict:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    embed_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda')
    print("BGE-large loaded on GPU")

    print("Computing Q&A relevance scores...")
    start = time.time()

    qa_relevance_scores = {}
    df_id_col = 'keydevid' if 'keydevid' in df.columns else TRANSCRIPT_ID_COL
    transcript_ids = df[df_id_col].values

    for i, tid in enumerate(transcript_ids):
        pairs = qa_pairs_dict.get(tid, [])
        if not pairs:
            qa_relevance_scores[tid] = np.nan
            continue

        sims = []
        for pair in pairs:
            q_emb = embed_model.encode([pair['question']])
            a_emb = embed_model.encode([pair['answer'][:1000]])  # truncate long answers
            sim = cosine_similarity(q_emb, a_emb)[0][0]
            sims.append(sim)

        qa_relevance_scores[tid] = np.mean(sims)

        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(transcript_ids)} ({(time.time()-start)/60:.1f}min)")

    df['qa_relevance'] = df[df_id_col].map(qa_relevance_scores)
    print(f"Q&A relevance — mean: {df['qa_relevance'].mean():.4f}")
    print(f"Q&A relevance scoring complete in {(time.time()-start)/60:.1f} minutes")
else:
    df['qa_relevance'] = np.nan
    print("Q&A relevance skipped — no component data available")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

BGE-large loaded on GPU
Computing Q&A relevance scores...
Q&A relevance — mean: nan
Q&A relevance scoring complete in 0.0 minutes


### Clarity Composite

In [15]:
from scipy.stats import zscore

# Determine which components are available
has_qa_rel = df['qa_relevance'].notna().sum() > 100

# Z-score each feature (handle NaN with fillna to median before z-scoring)
df['z_specificity'] = zscore(df['specificity'].fillna(df['specificity'].median()))
df['z_hedge'] = zscore(df['hedge_density'].fillna(df['hedge_density'].median()))
df['z_fog'] = zscore(df['modified_fog'].fillna(df['modified_fog'].median()))

if has_qa_rel:
    df['z_qa_relevance'] = zscore(df['qa_relevance'].fillna(df['qa_relevance'].median()))
    df['clarity_composite'] = (df['z_specificity'] - df['z_hedge'] - df['z_fog'] + df['z_qa_relevance']) / 4
    print("Clarity composite: 4 components (specificity, hedge, FOG, Q&A relevance)")
else:
    df['clarity_composite'] = (df['z_specificity'] - df['z_hedge'] - df['z_fog']) / 3
    print("Clarity composite: 3 components (specificity, hedge, FOG) — Q&A relevance unavailable")

print(f"Clarity composite — mean: {df['clarity_composite'].mean():.4f}, std: {df['clarity_composite'].std():.4f}")

Clarity composite: 3 components (specificity, hedge, FOG) — Q&A relevance unavailable
Clarity composite — mean: 0.0000, std: 0.5288


### Tercile Bucketing

In [16]:
# Extract year from date column — adapt column name
DATE_COL = 'announceddate'  # Based on available columns
df['call_year'] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.year

# Baseline period: 2010-2012
baseline = df[df['call_year'].between(2010, 2012)].copy()
test = df[df['call_year'] >= 2013].copy()

print(f"Baseline period (2010-2012): {len(baseline)} transcripts")
print(f"Test period (2013-2024): {len(test)} transcripts")

# Compute tercile cutpoints from baseline
sent_p33 = baseline['buyback_sentiment_mean'].quantile(0.333)
sent_p67 = baseline['buyback_sentiment_mean'].quantile(0.667)
clarity_p33 = baseline['clarity_composite'].quantile(0.333)
clarity_p67 = baseline['clarity_composite'].quantile(0.667)

print(f"\nBaseline sentiment cutpoints: P33={sent_p33:.4f}, P67={sent_p67:.4f}")
print(f"Baseline clarity cutpoints: P33={clarity_p33:.4f}, P67={clarity_p67:.4f}")

# Apply to full dataset
def bucket_sentiment(val):
    if pd.isna(val): return np.nan
    if val <= sent_p33: return 'Negative'
    if val <= sent_p67: return 'Neutral'
    return 'Positive'

def bucket_clarity(val):
    if pd.isna(val): return np.nan
    if val <= clarity_p33: return 'Low'
    if val <= clarity_p67: return 'Medium'
    return 'High'

df['sentiment_tercile'] = df['buyback_sentiment_mean'].apply(bucket_sentiment)
df['clarity_tercile'] = df['clarity_composite'].apply(bucket_clarity)

print(f"\nSentiment tercile distribution (full sample):")
print(df['sentiment_tercile'].value_counts())
print(f"\nClarity tercile distribution (full sample):")
print(df['clarity_tercile'].value_counts())

# 3x3 cross-tabulation
ct = pd.crosstab(df['sentiment_tercile'], df['clarity_tercile'])
print(f"\n3×3 Matrix — Cell Counts:")
print(ct)
print(f"\nMinimum cell count: {ct.min().min()}")

Baseline period (2010-2012): 841 transcripts
Test period (2013-2024): 10472 transcripts

Baseline sentiment cutpoints: P33=0.1929, P67=0.3989
Baseline clarity cutpoints: P33=-0.1448, P67=0.3085

Sentiment tercile distribution (full sample):
sentiment_tercile
Positive    4649
Neutral     3946
Negative    2714
Name: count, dtype: int64

Clarity tercile distribution (full sample):
clarity_tercile
Low       4201
Medium    3866
High      3246
Name: count, dtype: int64

3×3 Matrix — Cell Counts:
clarity_tercile    High   Low  Medium
sentiment_tercile                    
Negative            899   931     884
Neutral            1275  1329    1342
Positive           1072  1937    1640

Minimum cell count: 884


### EDA Charts

In [17]:
# 1. Buyback mention frequency
fig, ax = plt.subplots(figsize=(10, 5))
df['buyback_sentence_count'].clip(upper=30).hist(bins=30, ax=ax, color=COLORS['teal'], edgecolor='white', alpha=0.85)
ax.axvline(df['buyback_sentence_count'].median(), color=COLORS['red'], linestyle='--', linewidth=2,
           label=f"Median: {df['buyback_sentence_count'].median():.0f}")
ax.set_xlabel('Buyback Mentions per Call', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Buyback Mentions per Earnings Call', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/eda/buyback_mention_frequency.png', dpi=200, bbox_inches='tight')
plt.close()

# 2. Sentiment distributions (buyback vs full transcript)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['full_transcript_sentiment'].dropna(), bins=50, alpha=0.6, label='Full Transcript', color=COLORS['navy'], density=True)
axes[0].hist(df['buyback_sentiment_mean'].dropna(), bins=50, alpha=0.6, label='Buyback-Specific', color=COLORS['teal'], density=True)
axes[0].set_xlabel('FinBERT Sentiment Score')
axes[0].set_title('Sentiment: Full Transcript vs. Buyback-Specific', fontweight='bold')
axes[0].legend()
axes[0].axvline(0, color='gray', linestyle=':', alpha=0.5)

gap = df['sentiment_gap'].dropna()
if len(gap) > 0:
    axes[1].hist(gap, bins=50, color=COLORS['purple'], alpha=0.75, edgecolor='white')
    axes[1].axvline(gap.median(), color=COLORS['red'], linestyle='--', linewidth=2, label=f'Median: {gap.median():.3f}')
    axes[1].set_xlabel('Buyback Sentiment Gap')
    axes[1].set_title('Buyback Sentiment Gap Distribution', fontweight='bold')
    axes[1].legend()
plt.tight_layout()
plt.savefig('../outputs/eda/sentiment_distributions.png', dpi=200, bbox_inches='tight')
plt.close()

# 3. Sentiment by year
yearly = df.groupby('call_year').agg(
    mean_buyback=('buyback_sentiment_mean', 'mean'),
    count=('buyback_sentiment_mean', 'count')
).reset_index()
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(yearly['call_year'], yearly['mean_buyback'], marker='o', linewidth=2, color=COLORS['teal'], label='Buyback Sentiment')
ax1.set_xlabel('Year')
ax1.set_ylabel('Mean Sentiment')
ax1.set_title('Mean Buyback Sentiment by Year', fontweight='bold')
ax1.legend(loc='upper left')
ax1.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax2 = ax1.twinx()
ax2.bar(yearly['call_year'], yearly['count'], alpha=0.15, color=COLORS['teal'])
ax2.set_ylabel('N events', color='gray')
plt.tight_layout()
plt.savefig('../outputs/eda/sentiment_by_year.png', dpi=200, bbox_inches='tight')
plt.close()

# 4. Clarity distributions (4-panel)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(df['specificity'].dropna(), bins=5, color=COLORS['teal'], edgecolor='white', rwidth=0.85)
axes[0,0].set_title('Specificity Score (0-4)', fontweight='bold')
axes[0,1].hist(df['hedge_density'].dropna(), bins=40, color=COLORS['red'], edgecolor='white')
axes[0,1].set_title('Hedge Density', fontweight='bold')
axes[1,0].hist(df['modified_fog'].dropna().clip(0, 30), bins=40, color=COLORS['orange'], edgecolor='white')
if df['standard_fog'].notna().sum() > 0:
    axes[1,0].hist(df['standard_fog'].dropna().clip(0, 30), bins=40, color=COLORS['navy'], alpha=0.3)
    axes[1,0].legend(['Modified FOG', 'Standard FOG'])
axes[1,0].set_title('Modified FOG Index', fontweight='bold')
axes[1,1].hist(df['clarity_composite'].dropna(), bins=40, color=COLORS['purple'], edgecolor='white')
p33 = df['clarity_composite'].quantile(0.333)
p67 = df['clarity_composite'].quantile(0.667)
axes[1,1].axvline(p33, color=COLORS['red'], linestyle='--', label=f'P33: {p33:.2f}')
axes[1,1].axvline(p67, color=COLORS['green'], linestyle='--', label=f'P67: {p67:.2f}')
axes[1,1].legend()
axes[1,1].set_title('Clarity Composite', fontweight='bold')
plt.suptitle('Clarity Component Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/eda/clarity_distributions.png', dpi=200, bbox_inches='tight')
plt.close()

# 5. Clarity correlation matrix
corr_cols = ['specificity', 'hedge_density', 'modified_fog']
if has_qa_rel:
    corr_cols.append('qa_relevance')
corr_matrix = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0, square=True, linewidths=1, ax=ax)
ax.set_title('Clarity Component Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda/clarity_correlation_matrix.png', dpi=200, bbox_inches='tight')
plt.close()

# 6. FOG standard vs modified scatter
fig, ax = plt.subplots(figsize=(8, 6))
valid = df[['standard_fog', 'modified_fog']].dropna()
ax.scatter(valid['standard_fog'], valid['modified_fog'], alpha=0.1, s=8, color=COLORS['teal'])
ax.plot([0, 30], [0, 30], 'r--', alpha=0.5, label='y = x')
ax.set_xlabel('Standard Gunning FOG')
ax.set_ylabel('Modified FOG')
ax.set_title('FOG: Standard vs. Modified (Financial Vocab Excluded)', fontweight='bold')
ax.legend()
ax.set_xlim(0, 30); ax.set_ylim(0, 30)
plt.tight_layout()
plt.savefig('../outputs/eda/fog_standard_vs_modified.png', dpi=200, bbox_inches='tight')
plt.close()

# 7. 3x3 cell count heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', linewidths=1, ax=ax)
ax.set_title('3×3 Matrix — Sample Sizes per Cell', fontweight='bold')
ax.set_xlabel('Clarity Tercile')
ax.set_ylabel('Sentiment Tercile')
plt.tight_layout()
plt.savefig('../outputs/eda/matrix_cell_counts.png', dpi=200, bbox_inches='tight')
plt.close()

print("All EDA charts saved to ../outputs/eda/")

All EDA charts saved to ../outputs/eda/


### Save Scored Dataset

In [18]:
# Summary statistics
summary_cols = ['buyback_sentiment_mean', 'buyback_sentiment_p10', 'buyback_sentiment_min',
                'sentiment_gap', 'specificity', 'hedge_density', 'modified_fog',
                'standard_fog', 'clarity_composite', 'buyback_sentence_count']
if has_qa_rel:
    summary_cols.insert(-1, 'qa_relevance')

summary = df[summary_cols].describe().round(4)
summary.to_csv('../outputs/eda/summary_statistics.csv')
print("Summary Statistics:")
print(summary)

# Save scored dataset
output_cols = [c for c in df.columns if c != 'buyback_sentences']  # drop the list column
df[output_cols].to_csv('../data/interim/buyback_scored.csv', index=False)
print(f"\nSaved: ../data/interim/buyback_scored.csv ({len(df)} rows, {len(output_cols)} columns)")

# Print output inventory
print("\n" + "="*60)
for f in sorted(os.listdir('../outputs/eda')):
    print(f"  ../outputs/eda/{f}")

Summary Statistics:
       buyback_sentiment_mean  buyback_sentiment_p10  buyback_sentiment_min  \
count              11309.0000             11309.0000             11309.0000   
mean                   0.3641                 0.0518                -0.0430   
std                    0.2464                 0.3151                 0.4099   
min                   -0.7733                -0.9769                -0.9770   
25%                    0.1985                 0.0000                 0.0000   
50%                    0.3480                 0.0000                 0.0000   
75%                    0.5053                 0.0642                 0.0000   
max                    0.9541                 0.9541                 0.9541   

       sentiment_gap  specificity  hedge_density  modified_fog  standard_fog  \
count      4998.0000   11313.0000     11309.0000    11309.0000    11309.0000   
mean          0.1146       1.9108         0.0115       15.0578       15.1988   
std           0.2087       0